## **Imports**

In [1]:
import os
import json
import random
import numpy as np
import pandas as pd

from collections import Counter

import torch
from torch.utils.data import Dataset

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

In [2]:
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


## **Configurations**

In [3]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [4]:
CLAIMS_PATH = "/kaggle/input/datasets/anarvaaa/original-scifact-data/claims_train.jsonl"
CORPUS_PATH = "/kaggle/input/datasets/anarvaaa/original-scifact-data/corpus.jsonl"

## **Load Data**

In [57]:
claims = []

with open(CLAIMS_PATH, "r", encoding="utf-8") as f:
    for line in f:
        claims.append(json.loads(line))

print("Number of claims:", len(claims))
print(claims[0])

Number of claims: 809
{'id': 0, 'claim': '0-dimensional biomaterials lack inductive properties.', 'evidence': {}, 'cited_doc_ids': [31715818]}


In [58]:
corpus = []

with open(CORPUS_PATH, "r", encoding="utf-8") as f:
    for line in f:
        corpus.append(json.loads(line))

print("Number of documents:", len(corpus))
print(corpus[0])

Number of documents: 5183
{'doc_id': 4983, 'title': 'Microstructural development of human newborn cerebral white matter assessed in vivo by diffusion tensor magnetic resonance imaging.', 'abstract': ['Alterations of the architecture of cerebral white matter in the developing human brain can affect cortical development and result in functional disabilities.', 'A line scan diffusion-weighted magnetic resonance imaging (MRI) sequence with diffusion tensor analysis was applied to measure the apparent diffusion coefficient, to calculate relative anisotropy, and to delineate three-dimensional fiber architecture in cerebral white matter in preterm (n = 17) and full-term infants (n = 7).', 'To assess effects of prematurity on cerebral white matter development, early gestation preterm infants (n = 10) were studied a second time at term.', 'In the central white matter the mean apparent diffusion coefficient at 28 wk was high, 1.8 microm2/ms, and decreased toward term to 1.2 microm2/ms.', 'In the

In [59]:
doc_lookup = {}

for doc in corpus:
    doc_lookup[int(doc["doc_id"])] = doc

print("Documents in lookup:", len(doc_lookup))

Documents in lookup: 5183


In [60]:
example_doc_id = list(doc_lookup.keys())[0]

print("Doc ID:", example_doc_id)
print("Title:", doc_lookup[example_doc_id]["title"])
print("Abstract:")
print(doc_lookup[example_doc_id]["abstract"][:3])

Doc ID: 4983
Title: Microstructural development of human newborn cerebral white matter assessed in vivo by diffusion tensor magnetic resonance imaging.
Abstract:
['Alterations of the architecture of cerebral white matter in the developing human brain can affect cortical development and result in functional disabilities.', 'A line scan diffusion-weighted magnetic resonance imaging (MRI) sequence with diffusion tensor analysis was applied to measure the apparent diffusion coefficient, to calculate relative anisotropy, and to delineate three-dimensional fiber architecture in cerebral white matter in preterm (n = 17) and full-term infants (n = 7).', 'To assess effects of prematurity on cerebral white matter development, early gestation preterm infants (n = 10) were studied a second time at term.']


## **Load SciBERT**

In [61]:
MODEL_NAME = "allenai/scibert_scivocab_uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print(tokenizer)

BertTokenizer(name_or_path='allenai/scibert_scivocab_uncased', vocab_size=31090, model_max_length=1000000000000000019884624838656, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	104: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)


## **Get first 200 tokens Abstract**

In [62]:
def get_first_200_tokens(document_abstract):
    
    # Convert list of abstract sentences into one document
    document_text = " ".join(document_abstract)

    # Tokenize WITHOUT special tokens
    tokens = tokenizer.tokenize(document_text)

    # Keep first 200 SciBERT tokens
    first_200_tokens = tokens[:400]

    # Convert tokens back to text
    first_200_text = tokenizer.convert_tokens_to_string(
        first_200_tokens
    )

    return first_200_text

In [63]:
test_doc = corpus[0]

first_200 = get_first_200_tokens(
    test_doc["abstract"]
)

print(first_200)

Alterations of the architecture of cerebral white matter in the developing human brain can affect cortical development and result in functional disabilities. A line scan diffusion - weighted magnetic resonance imaging ( [UNK] ) sequence with diffusion tensor analysis was applied to measure the apparent diffusion coefficient, to calculate relative anisotropy, and to delineate three - dimensional fiber architecture in cerebral white matter in preterm ( n = 17 ) and full - term infants ( n = 7 ). To assess effects of prematurity on cerebral white matter development, early gestation preterm infants ( n = 10 ) were studied a second time at term. In the central white matter the mean apparent diffusion coefficient at 28 wk was high, 1. 8 microm2 / ms, and decreased toward term to 1. 2 microm2 / ms. In the posterior limb of the internal capsule, the mean apparent diffusion coefficients at both times were similar ( 1. 2 versus 1. 1 microm2 / ms ). Relative anisotropy was higher the closer birth

In [64]:
claims_with_evidence = [
    item for item in claims
    if item.get("evidence")
]

print("Original claims:", len(claims))
print("Claims with evidence:", len(claims_with_evidence))
print("Skipped:", len(claims) - len(claims_with_evidence))

Original claims: 809
Claims with evidence: 505
Skipped: 304


## **Create Training Examples**

In [65]:
examples = []

skipped_mixed = 0
missing_docs = 0

for item in claims_with_evidence:

    claim = item["claim"]

    for doc_id_str, evidence_list in item["evidence"].items():

        doc_id = int(doc_id_str)

        # Find document
        if doc_id not in doc_lookup:
            missing_docs += 1
            continue

        document = doc_lookup[doc_id]

        # Collect all evidence labels
        labels = set(
            evidence["label"]
            for evidence in evidence_list
        )

        # Aggregate sentence-level → document-level
        if labels == {"SUPPORT"}:
            document_label = "SUPPORT"

        elif labels == {"CONTRADICT"}:
            document_label = "CONTRADICT"

        else:
            # Both SUPPORT and CONTRADICT
            skipped_mixed += 1
            continue

        # Get first 200 SciBERT tokens of abstract
        abstract_prefix = get_first_200_tokens(
            document["abstract"]
        )

        examples.append({
            "claim_id": item["id"],
            "doc_id": doc_id,
            "claim": claim,
            "document_prefix": abstract_prefix,
            "label": document_label
        })

print("Final examples:", len(examples))
print("Skipped mixed-label examples:", skipped_mixed)
print("Missing documents:", missing_docs)

Final examples: 564
Skipped mixed-label examples: 0
Missing documents: 0


In [66]:
df = pd.DataFrame(examples)
print(df.shape)
df.head()

(564, 5)


,claim_id,doc_id,claim,document_prefix,label
0,2,13734012,1 in 5 million in UK have abnormal PrP positiv...,[UNK] To carry out a further survey of archive...,CONTRADICT
1,9,44265107,32% of liver transplantation programs required...,ContextChronic hepatitis C is the leading caus...,SUPPORT
2,12,33409100,40mg/day dosage of folic acid and 2mg/day dosa...,CONTEXT [UNK] plasma homocysteine levels are a...,SUPPORT
3,22,6490571,76-85% of people with severe mental disorder r...,CONTEXT Little is known about the extent or se...,SUPPORT
4,28,12670680,A T helper 2 cell (Th2) environment impedes di...,"In systemic lupus erythematosus ( SLE ), self ...",CONTRADICT


In [67]:
print(df["label"].value_counts())
print("\nPercentages:")
print(df["label"].value_counts(normalize=True) * 100)

label
SUPPORT       370
CONTRADICT    194
Name: count, dtype: int64

Percentages:
label
SUPPORT       65.602837
CONTRADICT    34.397163
Name: proportion, dtype: float64


## **Label Encoding**

In [68]:
label2id = {
    "CONTRADICT": 0,
    "SUPPORT": 1
}

id2label = {
    0: "CONTRADICT",
    1: "SUPPORT"
}

df["label_id"] = df["label"].map(label2id)

print(df[["label", "label_id"]].drop_duplicates())

        label  label_id
0  CONTRADICT         0
1     SUPPORT         1


## **Train-Val Split**

In [69]:
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=SEED,
    stratify=df["label_id"]
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print("Training examples:", len(train_df))
print("Validation examples:", len(val_df))

print("\nTraining distribution:")
print(train_df["label"].value_counts())

print("\nValidation distribution:")
print(val_df["label"].value_counts())

Training examples: 451
Validation examples: 113

Training distribution:
label
SUPPORT       296
CONTRADICT    155
Name: count, dtype: int64

Validation distribution:
label
SUPPORT       74
CONTRADICT    39
Name: count, dtype: int64


## **Dataset Class**

In [70]:
class SciFactDataset(Dataset):

    def __init__(
        self,
        dataframe,
        tokenizer,
        max_length=512
    ):
        self.data = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):

        row = self.data.iloc[idx]

        encoding = self.tokenizer(
            row["claim"],
            row["document_prefix"],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        item = {
            key: value.squeeze(0)
            for key, value in encoding.items()
        }

        item["labels"] = torch.tensor(
            row["label_id"],
            dtype=torch.long
        )

        return item

In [71]:
train_dataset = SciFactDataset(
    train_df,
    tokenizer,
    max_length=512
)

val_dataset = SciFactDataset(
    val_df,
    tokenizer,
    max_length=512
)

print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))

Train: 451
Validation: 113


## **Check**

In [72]:
sample = train_dataset[0]

print(sample.keys())
print("Input shape:", sample["input_ids"].shape)
print("Label:", sample["labels"].item())

dict_keys(['input_ids', 'token_type_ids', 'attention_mask', 'labels'])
Input shape: torch.Size([512])
Label: 0


In [73]:
print(
    tokenizer.decode(
        sample["input_ids"],
        skip_special_tokens=False
    )
)

[CLS] [UNK] produce less trimethylamine N - oxide from dietary I - carnitine than vegans. [SEP] Intestinal microbiota metabolism of choline and phosphatidylcholine produces trimethylamine ( TMA ), which is further metabolized to a proatherogenic species, trimethylamine - N - oxide ( TMAO ). [UNK] demonstrate here that metabolism by intestinal microbiota of dietary L - carnitine, a trimethylamine abundant in red meat, also produces TMAO and accelerates atherosclerosis in mice. [UNK] human subjects produced more TMAO than did vegans or vegetarians following ingestion of L - carnitine through a microbiota - dependent mechanism. The presence of specific bacterial taxa in human feces was associated with both plasma TMAO concentration and dietary status. Plasma L - carnitine levels in subjects undergoing cardiac evaluation ( n = 2, 595 ) predicted increased risks for both prevalent cardiovascular disease ( CVD ) and incident major adverse cardiac events ( myocardial infarction, stroke or dea

## **Train**

In [74]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label=id2label,
    label2id=label2id
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: allenai/scibert_scivocab_uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were ne

In [75]:
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(
        logits,
        axis=-1
    )

    accuracy = accuracy_score(
        labels,
        predictions
    )

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="macro",
        zero_division=0
    )

    return {
        "accuracy": accuracy,
        "precision_macro": precision,
        "recall_macro": recall,
        "f1_macro": f1
    }

In [78]:
training_args = TrainingArguments(
    output_dir="/kaggle/working/scibert_scifact_200",

    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=1e-5,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,

    num_train_epochs=5,
    weight_decay=0.01,
    load_best_model_at_end=True,

    metric_for_best_model="f1_macro",
    greater_is_better=True,

    logging_steps=50,
    save_total_limit=2,
    report_to="none",

    fp16=torch.cuda.is_available()
)

In [79]:
trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=train_dataset,
    eval_dataset=val_dataset,

    compute_metrics=compute_metrics
)

In [80]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,Precision Macro,Recall Macro,F1 Macro
1,No log,1.322076,0.628319,0.584993,0.582814,0.583684
2,1.269155,1.325952,0.637168,0.547619,0.522869,0.494710
3,1.269155,1.347481,0.575221,0.474396,0.481635,0.466562
4,1.065584,1.373264,0.557522,0.467916,0.474186,0.464962
5,1.065584,1.385093,0.548673,0.460504,0.467429,0.459026


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=145, training_loss=1.1009554106613686, metrics={'train_runtime': 163.2163, 'train_samples_per_second': 13.816, 'train_steps_per_second': 0.888, 'total_flos': 593315429836800.0, 'train_loss': 1.1009554106613686, 'epoch': 5.0})

## **Evaluation**

In [81]:
results = trainer.evaluate()

print(results)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{'eval_loss': 1.3216509819030762, 'eval_accuracy': 0.6283185840707964, 'eval_precision_macro': 0.5849928876244666, 'eval_recall_macro': 0.5828135828135828, 'eval_f1_macro': 0.5836842105263158, 'eval_runtime': 2.3227, 'eval_samples_per_second': 48.65, 'eval_steps_per_second': 1.722, 'epoch': 5.0}


In [82]:
predictions = trainer.predict(val_dataset)

logits = predictions.predictions
true_labels = predictions.label_ids

pred_labels = np.argmax(
    logits,
    axis=-1
)

print(
    classification_report(
        true_labels,
        pred_labels,
        target_names=[
            "CONTRADICT",
            "SUPPORT"
        ],
        digits=4
    )
)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


              precision    recall  f1-score   support

  CONTRADICT     0.4595    0.4359    0.4474        39
     SUPPORT     0.7105    0.7297    0.7200        74

    accuracy                         0.6283       113
   macro avg     0.5850    0.5828    0.5837       113
weighted avg     0.6239    0.6283    0.6259       113



In [83]:
cm = confusion_matrix(
    true_labels,
    pred_labels
)

print(cm)

[[17 22]
 [20 54]]


## **Save**

In [84]:
MODEL_SAVE_PATH = (
    "/kaggle/working/"
    "scibert_scifact_200_classifier"
)

trainer.save_model(MODEL_SAVE_PATH)
tokenizer.save_pretrained(MODEL_SAVE_PATH)

print("Model saved at:")
print(MODEL_SAVE_PATH)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved at:
/kaggle/working/scibert_scifact_200_classifier


In [85]:
preprocessing_config = {
    "document_token_limit": 200,
    "tokenizer": MODEL_NAME,
    "label2id": label2id,
    "id2label": id2label,
    "aggregation_rule": (
        "SUPPORT if all evidence sentences are SUPPORT; "
        "CONTRADICT if all evidence sentences are CONTRADICT; "
        "mixed-label documents skipped"
    )
}

with open(
    os.path.join(
        MODEL_SAVE_PATH,
        "preprocessing_config.json"
    ),
    "w"
) as f:
    json.dump(
        preprocessing_config,
        f,
        indent=4
    )

In [86]:
print("done")

done


In [87]:
!zip -r /kaggle/working/scibert_scifact_200_classifier.zip \
    /kaggle/working/scibert_scifact_200_classifier

  adding: kaggle/working/scibert_scifact_200_classifier/ (stored 0%)
  adding: kaggle/working/scibert_scifact_200_classifier/tokenizer_config.json (deflated 42%)
  adding: kaggle/working/scibert_scifact_200_classifier/model.safetensors (deflated 7%)
  adding: kaggle/working/scibert_scifact_200_classifier/training_args.bin (deflated 53%)
  adding: kaggle/working/scibert_scifact_200_classifier/tokenizer.json (deflated 71%)
  adding: kaggle/working/scibert_scifact_200_classifier/preprocessing_config.json (deflated 47%)
  adding: kaggle/working/scibert_scifact_200_classifier/config.json (deflated 52%)
